# getitem-back-add-at — worked example 3: Scatter-add backward matches autograd

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `getitem-back-add-at`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The manual scatter-add gradient should agree exactly with PyTorch autograd. Building a gather with a repeated index, back-propagating a scalar, and comparing `x.grad` to the hand-rolled `index_add_` result confirms the rule — including the crucial accumulation at repeated indices.

## Worked solution

We validate the hand-rolled backward against autograd.

1. Create a leaf tensor `x` with `requires_grad=True` and gather rows with a repeated index: `out = x[idx]`.
2. Reduce to a scalar with a weighted sum so the upstream gradient `grad_out` is a known tensor, then call `.backward()`. Autograd populates `x.grad` using its own scatter-add.
3. Independently compute the same gradient by hand: `grad_in = zeros_like(x)` then `index_add_(0, idx, grad_out)`, where `grad_out` is the gradient of our scalar with respect to `out`.
4. We assert the two match with `allclose`. Agreement proves the manual rule reproduces autograd's accumulation at the repeated index.

In [ ]:
import torch as t

t.manual_seed(2)
x = t.randn(4, 3, requires_grad=True)
idx = t.tensor([0, 2, 0])
w = t.tensor([[1.0, 1.0, 1.0], [2.0, 2.0, 2.0], [3.0, 3.0, 3.0]])

out = x[idx]
loss = (out * w).sum()
loss.backward()

grad_out = w  # d(loss)/d(out)
manual = t.zeros_like(x)
manual.index_add_(0, idx, grad_out)
print('autograd matches manual:', bool(t.allclose(x.grad, manual)))
print('row 0 accumulated:', x.grad[0].tolist())